# 02. DCA-Trie v1: Static Symbolic Filtering

**Purpose:** Implement and evaluate DCA-Trie v1 — KG path filtering at construction time using the symbolic TypeOracle.

**Changes from GCR baseline:** The trie is built from only those paths that pass both symbolic gates (type gate at terminal hop + range gate at all hops).

**Reference:** Chapter 3, §3.6, Algorithm 1.

**Key difference from old embedding-based design:** No encoder calls, no cosine similarity, no threshold τ. Admission uses TypeOracle symbolic gates.

In [ ]:
#@title 1. Setup & Environment
import sys
import os
import time
import json
import argparse
import warnings

warnings.filterwarnings(
    "ignore", category=UserWarning, module="transformers"
)

import torch

# GPU detection: auto-detect T4/L4/A100/H100, map to SM arch.
GPU_ARCH_MAP = {
    "A100": "80",
    "L4": "89",
    "T4": "75",
    "V100": "70",
    "H100": "90",
    "H200": "90",
    "RTX 4090": "89",
    "RTX 3090": "86",
}

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    gpu_arch = None
    for key, arch in GPU_ARCH_MAP.items():
        if key in gpu_name:
            gpu_arch = arch
            break
    has_a100 = "A100" in gpu_name
    print(f"GPU: {gpu_name}  |  VRAM: {gpu_mem:.1f} GB  |  SM arch: {gpu_arch}")
else:
    gpu_name = "None"
    gpu_mem = 0
    gpu_arch = None
    has_a100 = False
    print("WARNING: No GPU detected. Inference will be very slow.")

# Mount Google Drive (fallback gracefully if not on Colab).
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    IS_COLAB = True
except ImportError:
    IS_COLAB = False
    print("Not on Google Colab. Skipping Drive mount.")

# Clone repo (skip if already cloned).
REPO_DIR = (
    "/content/graph-constrained-reasoning"
    if IS_COLAB
    else os.getcwd()
)
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/Adjanour/graph-constrained-reasoning-dca {REPO_DIR}
    print(f"Cloned to {REPO_DIR}")
else:
    print(f"Repo already exists at {REPO_DIR}")

if IS_COLAB:
    os.chdir(REPO_DIR)
sys.path.insert(0, '.')

# Install pinned dependencies.
DEPS = [
    "transformers==4.44.2",
    "accelerate>=0.30.1",
    "datasets>=2.19.2",
    "marisa-trie>=1.2.0",
    "networkx>=3.0",
    "scikit-learn>=1.5.0",
    "tiktoken>=0.7.0",
    "sentencepiece>=0.2.0",
    "protobuf>=5.27.1",
]

print("Installing core dependencies...")
!pip install -q {' '.join(DEPS)}
print("Done.")

# Install flash-attn (optional, 3-tier fallback).
INSTALL_FLASH_ATTN = False
flash_attn_installed = False

if INSTALL_FLASH_ATTN and torch.cuda.is_available():
    py_ver = f"{sys.version_info.major}{sys.version_info.minor}"
    cuda_ver = torch.version.cuda.replace(".", "")
    torch_ver = torch.__version__.split("+")[0]
    torch_major_minor = ".".join(torch_ver.split(".")[:2])

    print(f"Python: {py_ver}  CUDA: {cuda_ver}  PyTorch: {torch_ver}")

    # Tier 1: pre-compiled wheel.
    wheel_url = (
        f"https://github.com/Dao-AILab/flash-attention/releases/"
        f"download/v2.7.3/flash_attn-2.7.3+cu{cuda_ver}"
        f"torch{torch_major_minor}cxx11abiFALSE-cp{py_ver}"
        f"-cp{py_ver}-linux_x86_64.whl"
    )
    print("Trying pre-compiled wheel...")
    ret = os.system(f"pip install -q '{wheel_url}' 2>/dev/null")
    if ret == 0:
        flash_attn_installed = True
        print("flash-attn installed from pre-compiled wheel.")
    else:
        print("Pre-compiled wheel not available.")

    # Tier 2: source install with ninja + arch targeting.
    if not flash_attn_installed:
        print("Falling back to source install with ninja...")
        !pip install -q ninja
        if gpu_arch:
            os.environ["FLASH_ATTN_CUDA_ARCHS"] = gpu_arch
            print(f"Targeting SM {gpu_arch} only (your GPU: {gpu_name})")
        n_cores = os.cpu_count() or 1
        print(f"Compiling with ninja ({n_cores} cores), ~10-15 min...")
        t0 = time.time()
        !pip install flash-attn --no-build-isolation
        elapsed = time.time() - t0
        try:
            import flash_attn
            flash_attn_installed = True
            print(
                f"flash-attn installed in {elapsed:.0f}s "
                f"(v{flash_attn.__version__})"
            )
        except ImportError:
            print(
                f"flash-attn install failed after {elapsed:.0f}s. "
                f"Using sdpa instead."
            )

if flash_attn_installed:
    print("flash_attention_2 available.")
else:
    print("Using sdpa attention (built into transformers, no install needed).")

# Verify key imports.
from tqdm import tqdm
from datasets import load_dataset
import numpy as np
from collections import defaultdict

from src.llms import get_registed_model
from src.trie import MarisaTrie
from src.qa_prompt_builder import PathGenerationWithAnswerPromptBuilder
import src.utils as utils

# Import symbolic TypeOracle.
sys.path.insert(0, os.path.join(os.getcwd(), 'notebooks'))
from type_oracle import TypeOracle, compute_type_irrelevance

print("\nAll imports verified.")
print(f"Working directory: {os.getcwd()}")

## 2. Configuration

All experiment parameters in one place. Adjust these before running.

In [ ]:
#@title 2. Configuration

# Model.
MODEL_PATH = "rmanluo/GCR-Meta-Llama-3.1-8B-Instruct"
MODEL_NAME = "gcr"

# Dataset.
DATA_PATH = "rmanluo"
DATASET = "RoG-webqsp"
SPLIT = "test"

# Decoding.
INDEX_LEN = 2
K = 5
GEN_MODE = "group-beam"
PROMPT_MODE = "zero-shot"
MAX_NEW_TOKENS = 256

# Attention implementation.
ATTN_IMPL = (
    "flash_attention_2" if (has_a100 and flash_attn_installed)
    else "sdpa"
)

# Sampling.
MAX_SAMPLES = 100

# Output (Google Drive).
DRIVE_BASE = (
    "/content/drive/MyDrive/GCR_Results"
    if IS_COLAB
    else "results/GenPaths"
)
FORCE_RERUN = False

# Derived settings.
TIMESTAMP = time.strftime("%Y%m%d_%H%M%S")
tag = "DCAv1-symbolic"

if IS_COLAB:
    OUTPUT_DIR = os.path.join(DRIVE_BASE, TIMESTAMP)
else:
    model_short = MODEL_PATH.split("/")[-1]
    postfix = f"{PROMPT_MODE}-{GEN_MODE}-k{K}-index_len{INDEX_LEN}-{tag}"
    OUTPUT_DIR = os.path.join(DRIVE_BASE, DATASET, model_short, SPLIT, postfix)

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("CONFIGURATION")
print(f"Model:          {MODEL_PATH}")
print(f"Dataset:        {DATA_PATH}/{DATASET} ({SPLIT})")
print(f"Max samples:    {MAX_SAMPLES or 'full'}")
print(f"Beam width:     {K}")
print(f"Index length:   {INDEX_LEN} hops")
print(f"Generation:     {GEN_MODE}")
print(f"Attention:      {ATTN_IMPL}")
print(f"GPU:            {gpu_name}")
print(f"Output:         {OUTPUT_DIR}")
print(f"Tag:            {tag}")
print(f"Gates:          type_gate (terminal hop) + range_gate (all hops)")

# Save config.
config = {
    "model_path": MODEL_PATH,
    "model_name": MODEL_NAME,
    "data_path": DATA_PATH,
    "dataset": DATASET,
    "split": SPLIT,
    "index_len": INDEX_LEN,
    "k": K,
    "gen_mode": GEN_MODE,
    "prompt_mode": PROMPT_MODE,
    "max_new_tokens": MAX_NEW_TOKENS,
    "max_samples": MAX_SAMPLES,
    "attn_impl": ATTN_IMPL,
    "gpu": gpu_name,
    "tag": tag,
    "timestamp": TIMESTAMP,
    "force_rerun": FORCE_RERUN,
}
with open(os.path.join(OUTPUT_DIR, "config.json"), "w") as f:
    json.dump(config, f, indent=2)
print(f"Config saved to {OUTPUT_DIR}/config.json")

## 3. Load LLM

In [ ]:
#@title 3. Load LLM
parser = argparse.ArgumentParser()
LLM = get_registed_model(MODEL_NAME)
LLM.add_args(parser)
args = parser.parse_args([])
args.model_path = MODEL_PATH
args.model_name = MODEL_NAME
args.k = K
args.generation_mode = GEN_MODE
args.attn_implementation = ATTN_IMPL
args.max_new_tokens = MAX_NEW_TOKENS
args.maximun_token = 4096

print(f"Loading {MODEL_PATH}...")
t0 = time.time()

try:
    model = LLM(args)
    model.prepare_for_inference()
    # Suppress sampling warnings for beam search.
    model.generation_cfg.temperature = None
    model.generation_cfg.top_p = None
    model.model.generation_config.temperature = None
    model.model.generation_config.top_p = None
except Exception as e:
    print(f"Error loading model: {e}")
    raise

load_time = time.time() - t0
n_params = sum(p.numel() for p in model.model.parameters()) / 1e6
vram_used = (
    torch.cuda.memory_allocated(0) / 1e9
    if torch.cuda.is_available()
    else 0
)

print(f"Loaded in {load_time:.1f}s")
print(f"Parameters: {n_params:.1f}M")
print(f"VRAM used:  {vram_used:.1f} GB")
print(f"Tokenizer vocab: {len(model.tokenizer)} tokens")
print("Ready.")

## 4. DCA-Trie v1: Build Symbolically Filtered Trie + Run Inference

In [ ]:
#@title 4. Build Symbolically Filtered Trie + Run Inference (Algorithm 1)

def build_dcav1_trie_sym(question_dict, tokenizer, index_path_length=2):
    """Build a symbolically filtered trie for DCA-Trie v1.

    Applies two symbolic gates (Algorithm 1, §3.6.2):
      1. Range gate at every hop — checks relation range constraint.
      2. Type gate at terminal hop only — checks entity type constraint.

    Returns:
        tuple: (trie, kept_paths, oracle, n_blocked) or
               (None, [], None, 0) if no paths survive.
    """
    oracle = TypeOracle.from_graph(question_dict["graph"])
    answer_types = oracle.infer_answer_types(question_dict["question"])

    entities = question_dict.get("q_entity", [])
    if "paths" in question_dict:
        paths_list = question_dict["paths"]
    else:
        g = utils.build_graph(question_dict["graph"])
        paths_list = utils.dfs(g, entities, index_path_length)

    if len(paths_list) == 0:
        return None, [], None, 0

    kept_paths = []
    n_type_blocked = 0
    n_range_blocked = 0

    for p in paths_list:
        h = len(p)
        admit = True

        # Gate 2 (range) — must pass at every hop.
        for _, relation, tail_entity in p:
            if not oracle.range_gate(relation, tail_entity):
                n_range_blocked += 1
                admit = False
                break

        # Gate 1 (type) — terminal hop only.
        if admit:
            terminal_entity = p[-1][2]
            if not oracle.type_gate(
                terminal_entity, answer_types, h, index_path_length
            ):
                n_type_blocked += 1
                admit = False

        if admit:
            kept_paths.append(p)

    if len(kept_paths) == 0:
        return None, [], oracle, n_type_blocked + n_range_blocked

    # Build trie from surviving paths.
    path_strings = [utils.path_to_string(p) for p in kept_paths]
    tokenized = tokenizer(
        path_strings, padding=False, add_special_tokens=False
    ).input_ids
    tokenized = [ids + [tokenizer.eos_token_id] for ids in tokenized]
    trie = MarisaTrie(tokenized, max_token_id=len(tokenizer) + 1)

    return trie, kept_paths, oracle, n_type_blocked + n_range_blocked


# Load dataset.
dataset = load_dataset(f"{DATA_PATH}/{DATASET}", split=SPLIT)
if MAX_SAMPLES is not None:
    dataset = dataset.select(range(min(MAX_SAMPLES, len(dataset))))
    print(f"Subsampled to {len(dataset)} questions")
else:
    print(f"Loaded {len(dataset)} questions")

# Set up output paths.
predictions_path = os.path.join(OUTPUT_DIR, "predictions.jsonl")

# Load existing results for resumption (checkpointing).
processed_ids = set()
if os.path.exists(predictions_path) and not FORCE_RERUN:
    with open(predictions_path, "r") as f:
        for line in f:
            try:
                processed_ids.add(json.loads(line)["id"])
            except (json.JSONDecodeError, KeyError):
                pass
    print(f"Resuming: {len(processed_ids)} already processed")

# Accumulators.
total_paths_before = 0
total_paths_after = 0
total_type_blocked = 0
total_range_blocked = 0
total_empty = 0
n_processed = 0

print(f"\nRunning DCA-Trie v1 (symbolic) on {len(dataset)} questions...")
t0 = time.time()

with open(predictions_path, "a" if processed_ids else "w") as fout:
    for data in tqdm(dataset, desc="DCA-Trie v1 (Symbolic)"):
        qid = data["id"]
        if qid in processed_ids:
            continue

        # Build DCA-Trie v1 with symbolic filtering.
        try:
            trie, kept_paths, oracle, n_blocked = build_dcav1_trie_sym(
                data, model.tokenizer, INDEX_LEN
            )
        except Exception as e:
            print(f"  [{qid}] Error building trie: {e}")
            continue

        if trie is None:
            total_empty += 1
            # Write empty result for checkpointing.
            result = {
                "id": qid,
                "question": data["question"],
                "prediction": "",
                "ground_truth": data["answer"],
                "n_paths_before": 0,
                "n_paths_after": 0,
                "n_blocked": n_blocked,
                "hit": False,
                "tag": tag,
            }
            fout.write(json.dumps(result) + "\n")
            fout.flush()
            n_processed += 1
            continue

        # Count paths before filtering.
        if "paths" in data:
            paths_before = len(data["paths"])
        else:
            g_tmp = utils.build_graph(data["graph"])
            paths_before = len(
                utils.dfs(g_tmp, data["q_entity"], INDEX_LEN)
            )
        total_paths_before += paths_before
        total_paths_after += len(kept_paths)
        total_type_blocked += n_blocked

        # Get ground truth paths.
        g = utils.build_graph(data["graph"])
        truth_paths = utils.get_truth_paths(
            data["q_entity"], data["a_entity"], g
        )
        ground_paths = [utils.path_to_string(p) for p in truth_paths]

        # Construct prompt and generate.
        path_strings = [utils.path_to_string(p) for p in kept_paths]
        prompt_builder = PathGenerationWithAnswerPromptBuilder(
            data["question"], path_strings, ground_paths
        )
        prompt = prompt_builder.build(mode=PROMPT_MODE)

        try:
            output_ids = model.model.generate(
                **model.tokenizer(
                    prompt,
                    return_tensors="pt",
                    add_special_tokens=True,
                ).to(model.model.device),
                max_new_tokens=MAX_NEW_TOKENS,
                num_beams=K,
                num_return_sequences=1,
                early_stopping=True,
                pad_token_id=model.tokenizer.eos_token_id,
            )
            pred_text = model.tokenizer.decode(
                output_ids[0],
                skip_special_tokens=True,
            )
        except Exception as e:
            print(f"  [{qid}] Generation error: {e}")
            pred_text = ""

        # Check hit.
        hit = any(
            a.lower() in pred_text.lower()
            for a in (data["answer"] if isinstance(data["answer"], list) else [data["answer"]])
        )

        result = {
            "id": qid,
            "question": data["question"],
            "prediction": pred_text,
            "ground_truth": data["answer"],
            "n_paths_before": paths_before,
            "n_paths_after": len(kept_paths),
            "n_blocked": n_blocked,
            "hit": hit,
            "tag": tag,
        }
        fout.write(json.dumps(result) + "\n")
        fout.flush()
        n_processed += 1

        if n_processed % 10 == 0:
            elapsed = time.time() - t0
            rate = n_processed / elapsed if elapsed > 0 else 0
            print(
                f"  [{n_processed}/{len(dataset)}] "
                f"paths_before={total_paths_before} "
                f"paths_after={total_paths_after} "
                f"empty={total_empty} "
                f"| {rate:.1f} q/s"
            )

elapsed_total = time.time() - t0

print(f"\nDone. Processed: {n_processed}")
print(f"Total paths before filtering: {total_paths_before}")
print(f"Total paths after filtering:  {total_paths_after}")
print(f"Type/range blocked:           {total_type_blocked}")
print(f"Empty tries (no paths):       {total_empty}")
print(f"Time: {elapsed_total:.1f}s")

## 5. Per-Gate Breakdown

In [ ]:
#@title 5. Per-Gate FNR Evaluation

print("PER-GATE FNR ANALYSIS (§3.8)")

# Re-run over the dataset to collect per-gate statistics.
val_dataset = load_dataset(f"{DATA_PATH}/{DATASET}", split=SPLIT)
if MAX_SAMPLES is not None:
    val_dataset = val_dataset.select(
        range(min(MAX_SAMPLES, len(val_dataset)))
    )

n_type_false_neg = 0
n_range_false_neg = 0
total_gold_paths = 0
n_questions = 0

t0 = time.time()

for data in tqdm(val_dataset, desc="FNR analysis"):
    g = utils.build_graph(data["graph"])
    truth_paths = utils.get_truth_paths(
        data["q_entity"], data["a_entity"], g
    )
    truth_strs = set(utils.path_to_string(p) for p in truth_paths)
    if len(truth_strs) == 0:
        continue
    total_gold_paths += len(truth_strs)
    n_questions += 1

    oracle = TypeOracle.from_graph(data["graph"])
    answer_types = oracle.infer_answer_types(data["question"])

    for p in truth_paths:
        h = len(p)

        # Check range gate on each hop.
        range_fail = False
        for _, relation, tail_entity in p:
            if not oracle.range_gate(relation, tail_entity):
                range_fail = True
                break

        # Check type gate on terminal hop.
        terminal_entity = p[-1][2]
        type_fail = not oracle.type_gate(
            terminal_entity, answer_types, h, INDEX_LEN
        )

        if type_fail:
            n_type_false_neg += 1
        if range_fail:
            n_range_false_neg += 1

elapsed_fnr = time.time() - t0

print(f"\nQuestions with gold paths: {n_questions}")
print(f"Total gold paths: {total_gold_paths}")
print()
print(
    f"Type gate false negatives:  {n_type_false_neg}  "
    f"(FNR_type = {n_type_false_neg / max(1, total_gold_paths):.4f})"
)
print(
    f"Range gate false negatives: {n_range_false_neg}  "
    f"(FNR_range = {n_range_false_neg / max(1, total_gold_paths):.4f})"
)

# Combined FNR (union bound, Eq. 3.17).
# A gold path is excluded if EITHER gate fails.
n_either_fail = 0
for data in val_dataset:
    g = utils.build_graph(data["graph"])
    truth_paths = utils.get_truth_paths(
        data["q_entity"], data["a_entity"], g
    )
    if not truth_paths:
        continue
    oracle = TypeOracle.from_graph(data["graph"])
    answer_types = oracle.infer_answer_types(data["question"])
    for p in truth_paths:
        if not p:
            continue
        range_fail = any(
            not oracle.range_gate(rel, tail) for _, rel, tail in p
        )
        type_fail = not oracle.type_gate(
            p[-1][2], answer_types, len(p), INDEX_LEN
        )
        if range_fail or type_fail:
            n_either_fail += 1

fnr_combined = n_either_fail / max(1, total_gold_paths)
print(
    f"Combined FNR (either gate): {n_either_fail}  "
    f"(FNR = {fnr_combined:.4f})"
)
print(f"Time: {elapsed_fnr:.1f}s")

# Save per-gate metrics.
fnr_metrics = {
    "n_questions": n_questions,
    "total_gold_paths": total_gold_paths,
    "n_type_false_neg": n_type_false_neg,
    "fnr_type": round(n_type_false_neg / max(1, total_gold_paths), 4),
    "n_range_false_neg": n_range_false_neg,
    "fnr_range": round(n_range_false_neg / max(1, total_gold_paths), 4),
    "n_either_fail": n_either_fail,
    "fnr_combined": round(fnr_combined, 4),
    "elapsed_s": round(elapsed_fnr, 1),
}
fnr_path = os.path.join(OUTPUT_DIR, "per_gate_fnr.json")
with open(fnr_path, "w") as f:
    json.dump(fnr_metrics, f, indent=2)
print(f"Saved to {fnr_path}")

## 6. Compare with GCR Baseline

In [ ]:
#@title 6. Compare with GCR Baseline

# Compute Hit@1 from saved predictions.
preds = []
if os.path.exists(predictions_path):
    with open(predictions_path, "r") as f:
        for line in f:
            try:
                preds.append(json.loads(line))
            except json.JSONDecodeError:
                pass

n_total = len(preds)
n_hits = sum(1 for p in preds if p.get("hit", False))
hit_at_1 = n_hits / max(1, n_total) * 100

avg_before = (
    sum(p.get("n_paths_before", 0) for p in preds) / max(1, n_total)
)
avg_after = (
    sum(p.get("n_paths_after", 0) for p in preds) / max(1, n_total)
)
pruning_ratio = (
    (1 - avg_after / max(1, avg_before)) * 100 if avg_before > 0 else 0
)

print("Comparison: GCR vs DCA-Trie v1 (Symbolic)")
print()
print(
    f"{'Metric':<25} {'GCR (baseline)':<18} {'DCA-Trie v1':<18}"
)
print("-" * 61)
print(f"{'Questions':<25} {n_total:<18} {n_total:<18}")
print(f"{'Hits@1':<25} {n_hits:<18} {n_hits:<18}")
print(f"{'Hit@1 (%)':<25} {'?':<18} {hit_at_1:.1f}%{'':>11}")
print(
    f"{'Faithfulness':<25} {'100%':<18} {'100% (Cond F)':<18}"
)
print(
    f"{'Avg paths/q':<25} {'~5000':<18} {avg_after:.0f}{'':>14}"
)
print(
    f"{'Pruning ratio':<25} {'0%':<18} {pruning_ratio:.1f}%{'':>13}"
)
print(f"{'SIR_type':<25} {'high':<18} {'~0 (type gate)':<18}")
print(f"{'SIR_traj':<25} {'high':<18} {'? (range gate)':<18}")

print()
print("Per-gate FNR:")
print(f"  Type gate FNR:  {fnr_metrics['fnr_type']}")
print(f"  Range gate FNR: {fnr_metrics['fnr_range']}")
print(f"  Combined FNR:   {fnr_metrics['fnr_combined']}")

print()
print("To measure decomposed SIR, run notebook 04_SIR_Evaluation.ipynb")
print("on the predictions.jsonl produced above.")

# Save comparison.
comparison = {
    "hit_at_1_pct": round(hit_at_1, 1),
    "n_hits": n_hits,
    "n_total": n_total,
    "avg_paths_before": round(avg_before, 1),
    "avg_paths_after": round(avg_after, 1),
    "pruning_ratio_pct": round(pruning_ratio, 1),
    "fnr_metrics": fnr_metrics,
    "time_s": round(elapsed_total, 1),
}
comparison_path = os.path.join(OUTPUT_DIR, "comparison.json")
with open(comparison_path, "w") as f:
    json.dump(comparison, f, indent=2)
print(f"\nComparison saved to {comparison_path}")